# Dataset Preparation for ML

In [ ]:
from IPython.display import display
import pandas as pd

from zebrafish_tensor_utils import (
    build_moa_labeled_tensor_dataset,
    build_tensor_embedding_2d,
    plot_tensor_embedding_2d,
)

from zebrafish_notebook_utils import (
    configure_full_dataframe_display,
    load_compound_image_condition_map_csv,
)

In [ ]:
configure_full_dataframe_display()

# Load the previously generated condition map snapshot instead of rebuilding it from the workbook and folders.
condition_df = load_compound_image_condition_map_csv()
condition_df.shape

In [ ]:
# User inputs
# selected_mechanisms:
#   list of normalized mechanism names from condition_df["mechanism_of_action"].
#   Use 2 or more classes depending on the comparison you want to study.
#   This default picks two similar pairs:
#   Pair 1: GABAAR_Antagonist vs GABAAR_NegativeAllostericModulator.
#   Pair 2: NMDAR_Activation vs NMDAR_Antagonist.
#   The goal is to create within-pair similarity but between-pair separation.
selected_mechanisms = [
    "GABAAR_Antagonist",
    "GABAAR_NegativeAllostericModulator",
    "NMDAR_Activation",
    "NMDAR_Antagonist",
]
# selected_concentrations:
#   subset of treatment concentration bands to include.
#   Typical options: ["high"], ["mid"], ["low"], or combinations such as ["high", "mid", "low"].
#   Water controls are added separately as class 0 and do not need to be listed here.
selected_concentrations = ["high", "mid", "low"]
# max_compounds_per_action:
#   maximum number of distinct compounds to keep for each mechanism class.
#   Use a small number for quick prototyping and a larger number for broader class coverage.
max_compounds_per_action = 2
# max_tensors_per_compound:
#   maximum number of condition tensors to keep per compound after filtering.
#   Increase this to use more runs / folders per compound, decrease it to balance classes faster.
max_tensors_per_compound = 4
# output_size:
#   target tensor size as (T, Z, Y, X) after cached loading, downsampling, and drift correction.
#   Use None in any position to keep the source dimension unchanged.
#   Examples: (10, 1, 32, 32), (10, 3, 64, 64), (None, 1, 64, 64).
output_size = (10, 3, 64, 64)
# only_active:
#   True keeps only folders not marked do_not_use / inactive.
#   False also includes non-active folders when present in the condition map.
only_active = True
# num_random_rotations:
#   number of random XY-rotation augmentations to add per base tensor.
#   0 means no augmented copies, 2 means original + 2 random rotations.
num_random_rotations = 2
# rotation_range_degrees:
#   maximum absolute XY rotation angle for augmentation.
#   Typical small values: 3.0 to 10.0 degrees.
rotation_range_degrees = 5.0
# normalize_global_drift:
#   True applies the LOESS-style global intensity drift correction during loading.
#   False keeps the raw intensity trend.
normalize_global_drift = True
# loess_frac:
#   smoothing fraction for the global intensity drift correction.
#   Smaller values follow local changes more closely; larger values smooth more aggressively.
#   Typical range: 0.1 to 0.4.
loess_frac = 0.25
# use_cache:
#   True reuses cached processed tensors from .tensor_cache when available.
#   False forces a rebuild from the source TIFFs.
use_cache = True
# use_tiff_cache:
#   True mirrors only the selected TIFF files into .tiff_cache before loading.
#   False reads directly from the mounted source paths.
use_tiff_cache = True
# random_seed:
#   seed used for reproducible dataset sampling and augmentation angles.
random_seed = 0
# embedding_method:
#   "pca" for a linear 2D projection, "umap" for a nonlinear neighborhood-preserving embedding.
embedding_method = "pca"
# umap_n_neighbors:
#   local neighborhood size for UMAP. Smaller values emphasize fine local structure,
#   larger values emphasize broader global structure.
umap_n_neighbors = 15
# umap_min_dist:
#   minimum spacing of points in the UMAP embedding.
#   Smaller values make tighter clusters, larger values spread clusters out more.
umap_min_dist = 0.1

In [ ]:
dataset = build_moa_labeled_tensor_dataset(
    condition_df=condition_df,
    selected_mechanisms=selected_mechanisms,
    selected_concentrations=selected_concentrations,
    max_compounds_per_action=max_compounds_per_action,
    max_tensors_per_compound=max_tensors_per_compound,
    output_size=output_size,
    only_active=only_active,
    num_random_rotations=num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    normalize_global_drift=normalize_global_drift,
    loess_frac=loess_frac,
    use_cache=use_cache,
    use_tiff_cache=use_tiff_cache,
    random_seed=random_seed,
)

In [ ]:
summary_df = pd.DataFrame(
    [
        {"metric": "n_examples", "value": int(dataset["tensors"].shape[0])},
        {"metric": "tensor_shape", "value": str(tuple(dataset["tensors"].shape))},
        {"metric": "label_tensor_shape", "value": str(tuple(dataset["labels"].shape))},
        {"metric": "timepoints_per_tensor", "value": int(dataset["tensors"].shape[1])},
        {"metric": "z_slices_per_tensor", "value": int(dataset["tensors"].shape[2])},
        {"metric": "image_height", "value": int(dataset["tensors"].shape[3])},
        {"metric": "image_width", "value": int(dataset["tensors"].shape[4])},
        {"metric": "n_classes_including_water", "value": int(len(dataset["label_map"]))},
    ]
)

label_map_df = pd.DataFrame(
    [{"label": label, "class_name": class_name} for label, class_name in dataset["label_map"].items()]
).sort_values("label").reset_index(drop=True)

display(summary_df)
display(label_map_df)

In [ ]:
display(dataset["metadata"].head(5))
display(dataset["metadata"].groupby(["label", "label_name"]).size().reset_index(name="n_examples"))

In [ ]:
# Flatten each tensor example and reduce it to 2D for quick class-separation inspection.
# Water remains class 0 in both the dataset labels and the plotted legend.
embedding_df = build_tensor_embedding_2d(
    tensors=dataset["tensors"],
    labels=dataset["labels"],
    label_map=dataset["label_map"],
    metadata=dataset["metadata"],
    method=embedding_method,
    random_state=random_seed,
    umap_n_neighbors=umap_n_neighbors,
    umap_min_dist=umap_min_dist,
)
display(embedding_df.head(5))

In [ ]:
plot_tensor_embedding_2d(
    embedding_df,
    title=f"{embedding_method.upper()} tensor embedding | water=0",
);